# Notebook 03 — Actual Production Data Processing

This notebook imports and parses your actual palm-level monthly nut count database (`Nut counting data.xlsx`) across 8 estates from 2019 to 2025, cleans spelling discrepancies, handles missing values, and calculates the monthly average yield per palm (Nuts/Palm).

## 1. Import Packages & Local Configuration

In [1]:
import sys
import os
import pandas as pd
import numpy as np
from pathlib import Path

# Ensure project root is in the path
sys.path.append(str(Path(os.getcwd()).parent))

import config
print("Processed Data Folder:", config.PROCESSED_DIR)

Processed Data Folder: D:\SLIIT\research_apk\coconut-yield-prediction-system\backend\Coconut_Yield_Prediction\processed


## 2. Load and Parse Actual Nut Counting Excel File

We load each sheet without headers and dynamically locate the month and year rows to ensure correct alignment across different sheets.

In [2]:
excel_path = config.RESOURCES_DIR / "Nut counting data.xlsx"
if not excel_path.exists():
    raise FileNotFoundError(f"Could not find actual yield Excel file at {excel_path}")

xls = pd.ExcelFile(excel_path)
records = []

for sheet in xls.sheet_names:
    df_sheet = pd.read_excel(excel_path, sheet_name=sheet, header=None)
    
    # Find the row containing months (indicated by J, F, M at start of month sequence)
    month_row_idx = -1
    for r in range(len(df_sheet)):
        row_vals = [str(x).strip() for x in df_sheet.iloc[r, 2:5]]
        if row_vals == ['J', 'F', 'M']:
            month_row_idx = r
            break
            
    if month_row_idx == -1:
        print(f"Warning: Could not find month headers in sheet: {sheet}")
        continue
        
    year_row_idx = month_row_idx - 1
    years_row = df_sheet.iloc[year_row_idx]
    
    # Extract years and their starting column indices
    year_cols = []
    for c in range(2, len(df_sheet.columns)):
        val = years_row.iloc[c]
        if pd.notna(val) and (isinstance(val, (int, float)) or str(val).isdigit()):
            year_cols.append((c, int(val)))
            
    months_row = df_sheet.iloc[month_row_idx]
    
    # Process palm data rows
    for r in range(month_row_idx + 1, len(df_sheet)):
        lsc = df_sheet.iloc[r, 0]
        palm = df_sheet.iloc[r, 1]
        
        # Skip empty rows
        if pd.isna(lsc) and pd.isna(palm):
            continue
            
        for i, (col_idx, year) in enumerate(year_cols):
            end_idx = year_cols[i+1][0] if i+1 < len(year_cols) else len(df_sheet.columns)
            
            for c in range(col_idx, end_idx):
                month_num = c - col_idx + 1
                val = df_sheet.iloc[r, c]
                month_char = months_row.iloc[c]
                
                # Skip if no month label exists (padding column)
                if pd.isna(month_char):
                    continue
                    
                records.append({
                    'District': sheet,  # Map sheet name to 'District' for consistency
                    'LSC': lsc,
                    'Palm_ID': palm,
                    'Year': year,
                    'Month': month_num,
                    'Nuts': val
                })

df_raw = pd.DataFrame(records)
print("Total raw observations parsed:", len(df_raw))
print(df_raw.head())

Total raw observations parsed: 12160
    District  LSC Palm_ID  Year  Month  Nuts
0  Makandura    1     480  2019      1   7.0
1  Makandura    1     480  2019      2   6.0
2  Makandura    1     480  2019      3   1.0
3  Makandura    1     480  2019      4   6.0
4  Makandura    1     480  2019      5  11.0


## 3. Data Cleaning and Imputation

We convert nut counts to numeric, identify missing data, and calculate the mean yield per estate, year, and month.

In [3]:
# Convert to float, force invalid entries to NaN
df_raw["Nuts"] = pd.to_numeric(df_raw["Nuts"], errors='coerce')

# Check for missing values before aggregation
print("Missing values in raw nut counts:", df_raw["Nuts"].isnull().sum())

# Impute missing raw counts using the palm's historical median for that month
# (or fallback to district-monthly median)
df_raw["Nuts"] = df_raw.groupby(["District", "Palm_ID", "Month"])["Nuts"].transform(lambda x: x.fillna(x.median()))
df_raw["Nuts"] = df_raw.groupby(["District", "Month"])["Nuts"].transform(lambda x: x.fillna(x.median()))

print("Missing values in raw nut counts after imputation:", df_raw["Nuts"].isnull().sum())

# Aggregate to get estate-monthly average nuts per palm
df_yield = df_raw.groupby(["District", "Year", "Month"])["Nuts"].agg(["mean", "count"]).reset_index()
df_yield = df_yield.rename(columns={"mean": "Avg_Nuts_Palm", "count": "Palms_Count"})
df_yield["District"] = df_yield["District"].str.strip()

print("\nDistrict-monthly yield shape:", df_yield.shape)
print(df_yield.head())

Missing values in raw nut counts: 43
Missing values in raw nut counts after imputation: 0

District-monthly yield shape: (608, 5)
     District  Year  Month  Avg_Nuts_Palm  Palms_Count
0  Henwalawwa  2019      1           4.60           20
1  Henwalawwa  2019      2           2.10           20
2  Henwalawwa  2019      3           1.55           20
3  Henwalawwa  2019      4           3.75           20
4  Henwalawwa  2019      5           6.80           20


## 4. Standardize Estate Names & Save Output

In [4]:
# Verify that all estate names match the config estates exactly
config_estates = list(config.ESTATES.keys())
print("Config estates:", config_estates)
print("Yield dataset estates:", df_yield["District"].unique())

# Save clean output
out_path = config.PROCESSED_DIR / "production_clean.csv"
df_yield.to_csv(out_path, index=False)
print(f"\n[SUCCESS] Cleaned production dataset saved to {out_path.name}")

Config estates: ['Makandura', 'Mellawa', 'Muruthenge', 'Hewana', 'Saliya', 'Maliga estate', 'Henwalawwa', 'Jawatta']
Yield dataset estates: ['Henwalawwa' 'Hewana' 'Jawatta' 'Makandura' 'Maliga estate' 'Mellawa'
 'Muruthenge' 'Saliya']

[SUCCESS] Cleaned production dataset saved to production_clean.csv
